In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# .env 파일에서 API 키 로드
load_dotenv()

# OpenAI LLM 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ 환경설정 완료")

## Weekend Mission: Education Agent (필수 요구 반영)

- **노드 4개**: `analyze_need` → (`run_search` | `skip_search`) → `synthesize`
- **조건부 엣지 1개**: 질문에 따라 웹 검색 경로와 생략 경로로 분기
- **Tool 1개**: `educational_web_lookup` (DuckDuckGo 텍스트 검색)

아래 셀을 순서대로 실행하세요 (상단 환경 셀에서 `llm`이 이미 초기화되어 있어야 합니다).

In [ ]:
import json
import re
from typing import Literal

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from duckduckgo_search import DDGS


# ── Tool (필수: 외부 검색) ─────────────────────────────────────────
@tool
def educational_web_lookup(query: str, max_results: int = 3) -> str:
    """교육·학습 질문 보조용 웹 검색. 최신 정보, 정의, 통계 확인에 사용합니다."""
    query = (query or "").strip()
    if not query:
        return "(검색어가 비어 있습니다.)"
    lines: list[str] = []
    try:
        with DDGS() as ddgs:
            found = list(ddgs.text(query, max_results=max_results))
        for i, item in enumerate(found, 1):
            title = item.get("title", "").strip()
            body = (item.get("body") or "").strip()[:400]
            href = item.get("href", "").strip()
            lines.append(f"{i}. {title}\n   {body}\n   {href}")
    except Exception as e:
        return f"(검색 중 오류: {e})"
    return "\n\n".join(lines) if lines else "(검색 결과 없음)"


class EducationState(TypedDict):
    user_question: str
    needs_search: bool
    search_query: str
    search_results: str
    final_answer: str


def _parse_router_json(raw: str) -> tuple[bool, str]:
    raw = raw.strip()
    m = re.search(r"\{[\s\S]*\}", raw)
    if m:
        raw = m.group(0)
    data = json.loads(raw)
    needs = bool(data.get("needs_search"))
    sq = (data.get("search_query") or "").strip()
    return needs, sq


# Node 1: 질문 분석 + 검색 필요 여부 판단 (조건부 엣지의 입력)
def analyze_need(state: EducationState) -> EducationState:
    q = state["user_question"]
    print(f"\n📌 [Education] 질문 분석: {q[:80]}{'…' if len(q) > 80 else ''}")

    messages = [
        SystemMessage(
            content="""당신은 교육 에이전트의 라우터입니다.
학습자 질문을 보고 웹 검색이 필요하면 needs_search를 true로 두세요.

검색이 **필요한** 예: 최신 시험 일정/정책, 구체적 수치·통계, 최근 뉴스, 검증이 필요한 사실, 특정 용어의 공식 정의가 불확실한 경우.
검색이 **불필요한** 예: 일반 개념 설명, 학습 방법 조언, 순수 수학·코딩 아이디어(최신 정보 없이 답 가능).

반드시 JSON만 한 덩어리로 출력하세요. 키는 needs_search(bool), search_query(str) 두 개입니다.
needs_search가 false이면 search_query는 빈 문자열로 두세요."""
        ),
        HumanMessage(content=q),
    ]
    raw = llm.invoke(messages).content.strip()
    try:
        needs, sq = _parse_router_json(raw)
    except (json.JSONDecodeError, TypeError, ValueError):
        needs, sq = True, q
    state["needs_search"] = needs
    state["search_query"] = sq if sq else q
    print(f"   → needs_search={needs}, search_query={(state['search_query'] or '')[:60]}…")
    return state


# Node 2a: Tool을 사용한 웹 조사
def run_search(state: EducationState) -> EducationState:
    print("🔎 웹 검색 노드 실행…")
    tool_out = educational_web_lookup.invoke(
        {"query": state["search_query"], "max_results": 4}
    )
    state["search_results"] = (
        tool_out if isinstance(tool_out, str) else str(tool_out)
    )
    print("✅ 검색 완료")
    return state


# Node 2b: 검색 생략 분기
def skip_search(state: EducationState) -> EducationState:
    print("⏭️ 검색 생략 분기 (개념 위주 답변 예정)")
    state["search_results"] = "(웹 검색 생략됨)"
    return state


def _route_after_analyze(state: EducationState) -> Literal["run_search", "skip_search"]:
    return "run_search" if state.get("needs_search") else "skip_search"


# Node 3: 최종 학습 답변 생성
def synthesize(state: EducationState) -> EducationState:
    print("📝 최종 답변 생성 중…")
    q = state["user_question"]
    ctx = state["search_results"]
    messages = [
        SystemMessage(
            content="""당신은 친절한 교육 도우미입니다. 한국어로 답변하세요.
아래에 웹 검색 요약이 있으면 활용하되, 출처가 불명확하면 단정하지 마세요.
마크다운으로 읽기 좋게 구조화하세요."""
        ),
        HumanMessage(
            content=f"질문:\n{q}\n\n참고 자료(검색 또는 생략 표시):\n{ctx}\n\n위를 바탕으로 학습자에게 답해 주세요."
        ),
    ]
    resp = llm.invoke(messages)
    state["final_answer"] = resp.content
    print("✅ Education Agent 완료")
    return state


# 그래프 조립: 노드 4개 + 조건부 엣지 1개
education_graph = StateGraph(EducationState)
education_graph.add_node("analyze_need", analyze_need)
education_graph.add_node("run_search", run_search)
education_graph.add_node("skip_search", skip_search)
education_graph.add_node("synthesize", synthesize)

education_graph.set_entry_point("analyze_need")
education_graph.add_conditional_edges(
    "analyze_need",
    _route_after_analyze,
    {"run_search": "run_search", "skip_search": "skip_search"},
)
education_graph.add_edge("run_search", "synthesize")
education_graph.add_edge("skip_search", "synthesize")
education_graph.add_edge("synthesize", END)

education_app = education_graph.compile()

print("✅ Education Agent 그래프 컴파일 완료")
print("   START → analyze_need → [조건부] → run_search | skip_search → synthesize → END")

In [ ]:
# 실행 예시: 질문만 바꿔서 테스트하세요.

EDU_QUESTION = "2025년 대한민국 수능 영어 과목의 공식 일정 알려줘"  # 최신 정보 필요 → 검색 분기 가능
# EDU_QUESTION = "버블 정렬이 뭐야? 학습 순서 알려줘"  # 개념형 → 검색 생략 분기 가능

edu_state: EducationState = {
    "user_question": EDU_QUESTION,
    "needs_search": False,
    "search_query": "",
    "search_results": "",
    "final_answer": "",
}

print("=" * 60)
edu_result = education_app.invoke(edu_state)
print("=" * 60)
print("\n## 교육 에이전트 답변\n")
print(edu_result["final_answer"])